# My first SCF
After some time and effort, this is the first SCF calculation performed natively in Oatmeal. No preintegral evaluation, no nothing, only numpy's array operations and some scipy functions. 

In [1]:
from py_mods.src.integrals.Basis import (
    construct_basis_from_lists,
    S_basis_set,
    T_basis_set,
    V_basis_set,
    eri_basis_set,
)
from py_mods.src.integrals.CGTO import (
    create_CGTOClass,
    generate_angular_momentum_projections,
)
from py_mods.src.SCF.types import CSUHFContext
from py_mods.src.MP2.CSMP2 import CS_MP2
from py_mods.src.SCF.CSUHF import CS_UHF

import numpy as np

First, we build the basis set from positions, contraction coefficients, and atom charges. 

In [2]:
l = 0

r1 = np.array([0.0, 0.0, 0.0])
r2 = np.array([0.0, 0.0, 1.4])

H_exps = np.array([3.42525091, 0.62391373, 0.16885540])
H_coeffs = np.array([0.15432897, 0.53532814, 0.44463454])

atom_pos = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 1.4]])
atom_charges = np.array([1.0, 1.0])

# Construct the CGTO
H_1s_1 = create_CGTOClass(r1, H_exps, H_coeffs, l)
H_1p_1 = create_CGTOClass(r1, H_exps, H_coeffs, l + 1)
H_1d_1 = create_CGTOClass(r1, H_exps, H_coeffs, l + 2)
H_1s_2 = create_CGTOClass(r2, H_exps, H_coeffs, l)
H_1p_2 = create_CGTOClass(r2, H_exps, H_coeffs, l + 1)
H_1d_2 = create_CGTOClass(r2, H_exps, H_coeffs, l + 2)

bs = construct_basis_from_lists(
    [H_1s_1, H_1p_1, H_1s_2, H_1p_2], atom_pos, atom_charges
)

Then we evaluate all the necessary integrals: overlap, kinetic, nuclear attraction, and electron repulsion integrals:

In [3]:
S_bs = S_basis_set(bs)
T_bs = T_basis_set(bs)
V_bs = V_basis_set(bs)

In [4]:
eri_bs = eri_basis_set(bs)

And finally we call the SCF routine:

In [5]:
# RHF Energy PySCF: -1.834917848399617
# MP2 Energy PySCF: -1.858221693998765

rhf_ref_energy = -1.834917848399617
mp2_ref_energy = -1.858221693998765

H2_cxt = CSUHFContext(S_bs, T_bs, V_bs, eri_bs, 2)
H2_results = CS_UHF(H2_cxt)
H2_MP2_results = CS_MP2(H2_results)

print(
    f"Final RHF Energy: {H2_results.E_UHF.real:12.8f}, reference: {rhf_ref_energy:12.8f}, difference: {H2_results.E_UHF.real - rhf_ref_energy:4e}"
)
print(
    f"Final MP2 Energy: {H2_MP2_results.E_MP2.real:12.8f}, reference: {mp2_ref_energy:12.8f}, difference: {H2_MP2_results.E_MP2.real - mp2_ref_energy:4e}"
)

Final RHF Energy:  -1.83491785, reference:  -1.83491785, difference: -5.151435e-14
Final MP2 Energy:  -1.85822169, reference:  -1.85822169, difference: -6.195044e-14


Which is exactly the same as the reference. It is slow, yes, but it works. And I am not going to lie, I am quite happy about it